# Zimbabwe Used Vehicle Market Analysis
**Author:** Tatenda Alistair Gonzo  
**Snapshot date:** 5 November 2026  
**Tools:** Python, Pandas, NumPy and Matplotlib

This notebook explores 21 publicly advertised vehicle listing observations. It demonstrates data inspection, cleaning checks, descriptive statistics and visualisation. The sample is small, non-random and heavily weighted toward Toyota Axio listings; it is not representative of Zimbabwe's full vehicle market.

## 1. Load libraries and data
Run from the repository root in Jupyter. In Google Colab, upload the CSV when prompted.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

DATA_PATH = Path('data/used_vehicles.csv')
if not DATA_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError('Please upload used_vehicles.csv')
        DATA_PATH = Path(csv_files[0])
    except ImportError as exc:
        raise FileNotFoundError('Run this notebook from the repository root so data/used_vehicles.csv can be found.') from exc

vehicles = pd.read_csv(DATA_PATH)
print(f'Loaded {len(vehicles)} observations and {len(vehicles.columns)} columns.')
display(vehicles.head())

## 2. Inspect structure and data types

In [ ]:
print('Dataset shape:', vehicles.shape)
display(vehicles.dtypes.to_frame('data_type'))
display(vehicles.isna().sum().to_frame('missing_values'))

## 3. Prepare analysis fields
Original columns are retained. Missing values are not filled with invented values.

In [ ]:
for column in ['year', 'price', 'mileage_km']:
    vehicles[column] = pd.to_numeric(vehicles[column], errors='coerce')
vehicles['date_collected'] = pd.to_datetime(vehicles['date_collected'], errors='coerce')
vehicles['currency'] = vehicles['currency'].astype('string').str.upper()
usd_vehicles = vehicles.loc[vehicles['currency'].eq('USD')].copy()
print('USD observations:', len(usd_vehicles))
print('Duplicate project IDs:', vehicles['listing_id'].duplicated().sum())
display(usd_vehicles[['listing_id', 'make', 'model', 'year', 'price', 'mileage_km']].head())

## 4. Descriptive statistics
These are sample-level summaries, not estimates for the whole Zimbabwe market.

In [ ]:
display(usd_vehicles[['price', 'year', 'mileage_km']].describe().round(2))
print('Median asking price (USD):', usd_vehicles['price'].median())
print('Median mileage (km):', usd_vehicles['mileage_km'].median())
print('\nCounts by make:')
display(vehicles['make'].value_counts(dropna=False))
print('\nCounts by model:')
display(vehicles['model'].value_counts(dropna=False))

## 5. Visualise asking-price distribution

In [ ]:
VISUALS_DIR = Path('visuals')
VISUALS_DIR.mkdir(exist_ok=True)
plt.figure(figsize=(8, 5))
plt.hist(usd_vehicles['price'].dropna(), bins=8, edgecolor='black')
plt.title('Distribution of Advertised Asking Prices')
plt.xlabel('Asking price (USD)')
plt.ylabel('Number of observations')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'price_distribution.png', dpi=160)
plt.show()

## 6. Compare median asking price by model year
Small groups should be interpreted cautiously.

In [ ]:
price_by_year = usd_vehicles.groupby('year')['price'].median().sort_index()
plt.figure(figsize=(9, 5))
price_by_year.plot(kind='bar', edgecolor='black')
plt.title('Median Asking Price by Model Year')
plt.xlabel('Model year')
plt.ylabel('Median asking price (USD)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'median_price_by_year.png', dpi=160)
plt.show()

## 7. Explore mileage and asking price
A correlation is descriptive only and does not establish causation.

In [ ]:
mileage_price = usd_vehicles.dropna(subset=['mileage_km', 'price'])
plt.figure(figsize=(8, 5))
plt.scatter(mileage_price['mileage_km'], mileage_price['price'], alpha=0.8)
plt.title('Mileage vs Advertised Asking Price')
plt.xlabel('Mileage (km)')
plt.ylabel('Asking price (USD)')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'mileage_vs_price.png', dpi=160)
plt.show()
if len(mileage_price) >= 3:
    print('Sample Pearson correlation:', round(mileage_price['mileage_km'].corr(mileage_price['price']), 3))

## 8. Review missing values and possible duplicates
Potential duplicates are flagged for human review. Matching attributes do not prove two result cards refer to the same advert.

In [ ]:
quality = pd.DataFrame({'missing_count': vehicles.isna().sum(), 'missing_percent': (vehicles.isna().mean() * 100).round(1)}).sort_values('missing_percent', ascending=False)
display(quality)
plt.figure(figsize=(9, 5))
quality['missing_count'].sort_values().plot(kind='barh', edgecolor='black')
plt.title('Missing Values by Column')
plt.xlabel('Missing observations')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'missing_values.png', dpi=160)
plt.show()
duplicate_fields = ['make', 'model', 'year', 'price', 'mileage_km']
possible_duplicate_mask = vehicles.duplicated(subset=duplicate_fields, keep=False)
possible_duplicates = vehicles.loc[possible_duplicate_mask, ['listing_id', *duplicate_fields, 'source_name']].sort_values(duplicate_fields)
display(possible_duplicates)

## 9. Summary and limitations
The dataset is a small convenience sample dominated by Toyota Axio listings. Asking prices are not confirmed sale prices. Source URLs point to results/category pages, not individual adverts. Rerun the analysis after source-level verification and dataset expansion before drawing stronger conclusions.

In [ ]:
print(f'Observations analysed: {len(vehicles)}')
print(f'Median asking price: US${usd_vehicles["price"].median():,.0f}')
print(f'Median advertised mileage: {usd_vehicles["mileage_km"].median():,.0f} km')
print('Most common make:', vehicles['make'].mode().iat[0] if not vehicles['make'].mode().empty else 'N/A')
print('Potential duplicate rows flagged:', int(possible_duplicate_mask.sum()))
print('Charts saved to:', VISUALS_DIR.resolve())